In [1]:
import torch

x = torch.tensor(1.0, requires_grad=True) #指定需要计算梯度
y = torch.tensor(1.0, requires_grad=True) #指定需要计算梯度
v = 3*x+4*y
u = torch.square(v)
z = torch.log(u)

z.backward() #反向传播求梯度

print("x grad:", x.grad)
print("y grad:", y.grad)

x grad: tensor(0.8571)
y grad: tensor(1.1429)


### 手动实现上面的过程

In [ ]:
import math

# 定义一个 Tensor 类
class Tensor:
    def __init__(self, data, requires_grad=False):
        self.data = float(data)
        self.requires_grad = requires_grad
        self.grad = 0.0

        # 反向传播用
        self._backward = lambda: None # 相当于占位符，保证每个节点都有 _backward 可调用
        self._prev = []

# 定义算子（forward + backward）
def linear_3x_4y(x, y):
    v = Tensor(3*x.data + 4*y.data, requires_grad=True)
    v._prev = [x, y]

    def _backward():  # 针对特定运算节点生成对应的“真实反传规则”
        if x.requires_grad:
            x.grad += 3 * v.grad
        if y.requires_grad:
            y.grad += 4 * v.grad

    v._backward = _backward # 将函数赋值给节点的原占位属性
    return v

def square(v):
    u = Tensor(v.data ** 2, requires_grad=True)
    u._prev = [v]

    def _backward():
        if v.requires_grad:
            v.grad += 2 * v.data * u.grad

    u._backward = _backward
    return u

def log(u):
    z = Tensor(math.log(u.data), requires_grad=True)
    z._prev = [u]

    def _backward():
        if u.requires_grad:
            u.grad += (1 / u.data) * z.grad

    z._backward = _backward
    return z

# ===== forward =====
x = Tensor(1.0, requires_grad=True)
y = Tensor(1.0, requires_grad=True)

v = linear_3x_4y(x, y)
u = square(v)
z = log(u)

# ===== backward =====
z.grad = 1.0
z._backward()
u._backward()
v._backward()

print("x.grad =", f"{x.grad:.5f}")
print("y.grad =", f"{y.grad:.5f}")

x.grad = 0.85714
y.grad = 1.14286
